# SoulBites AI Analytics Model Training

This notebook covers the dataset generation, feature engineering, model training, evaluation, and serialization of the two machine learning models required for the SoulBites Chef Analytics platform:
1. **AI Model 1: Ingredient Estimation** (Multi-Output Regression with Portion size multipliers)
2. **AI Model 2: Dish Demand Forecasting** (Time Series/Regression Forecasting with Indian Dishes)

---

In [ ]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

print("Libraries imported successfully!")

## 1. AI Model 1: Ingredient Estimation (Subscription Planning)

When a user subscribes to a meal plan, the chef needs to plan ingredient purchases for the upcoming subscription period.

### Features:
- `num_subscribers`: Count of subscribers for a meal plan (integer)
- `duration_days`: Subscription length (7, 15, or 30 days)
- `portion_size_multiplier`: Individual subscriber appetite modifier (e.g. 0.8x for light eaters, 1.0x for standard, 1.25x for large portions, 1.5x for heavy portions)
- `meal_type`: Categorical (Veg Meal, Protein Meal, Family Meal)

### Targets (Multi-Output):
- `Rice_qty` (kg)
- `Tomato_qty` (kg)
- `Onion_qty` (kg)
- `Chicken_qty` (kg)
- `Paneer_qty` (kg)

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# 1. Generate Synthetic Subscriptions Dataset with portion size multipliers
num_samples = 1500
subscribers = np.random.randint(5, 60, size=num_samples)
durations = np.random.choice([7, 15, 30], size=num_samples)
meal_types = np.random.choice(['Veg Meal', 'Protein Meal', 'Family Meal'], size=num_samples)
portion_multipliers = np.random.choice([0.8, 1.0, 1.2, 1.5], size=num_samples, p=[0.2, 0.5, 0.2, 0.1])

# Define base ingredient requirements per serving (kg per serving)
recipe_bases = {
    'Veg Meal':     [0.15, 0.10, 0.08, 0.00, 0.12],
    'Protein Meal': [0.10, 0.08, 0.06, 0.20, 0.00],
    'Family Meal':  [0.50, 0.30, 0.25, 0.40, 0.20]
}

targets = []
for i in range(num_samples):
    m_type = meal_types[i]
    subs = subscribers[i]
    days = durations[i]
    mult = portion_multipliers[i]
    
    # Base total calculation scales by portion_multiplier
    base_needs = np.array(recipe_bases[m_type]) * subs * days * mult
    
    # Add normal noise representing kitchen waste (5% standard deviation)
    noise = np.random.normal(1.0, 0.05, size=5)
    actual_needs = np.maximum(0, base_needs * noise)
    targets.append(actual_needs)

targets = np.array(targets)

# Create DataFrame
sub_df = pd.DataFrame({
    'num_subscribers': subscribers,
    'duration_days': durations,
    'portion_size_multiplier': portion_multipliers,
    'meal_type': meal_types,
    'Rice_qty': targets[:, 0],
    'Tomato_qty': targets[:, 1],
    'Onion_qty': targets[:, 2],
    'Chicken_qty': targets[:, 3],
    'Paneer_qty': targets[:, 4]
})

print("Sample Subscriptions Dataset:")
print(sub_df.head())
sub_df.to_csv('synthetic_subscriptions.csv', index=False)
print("Dataset saved to synthetic_subscriptions.csv")

In [ ]:
# 2. Prepare Features (One-Hot Encode Categoricals)
X = pd.get_dummies(sub_df[['num_subscribers', 'duration_days', 'portion_size_multiplier', 'meal_type']], columns=['meal_type'])
y = sub_df[['Rice_qty', 'Tomato_qty', 'Onion_qty', 'Chicken_qty', 'Paneer_qty']]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train Multi-Output Random Forest Regressor
ingredient_model = RandomForestRegressor(n_estimators=50, random_state=42)
ingredient_model.fit(X_train, y_train)

# Evaluate
predictions = ingredient_model.predict(X_test)
print("--- Ingredient Estimation Model Evaluation ---")
for idx, col in enumerate(y.columns):
    mae = mean_absolute_error(y_test[col], predictions[:, idx])
    r2 = r2_score(y_test[col], predictions[:, idx])
    print(f"{col:12} -> MAE: {mae:.2f} kg, R^2 Score: {r2:.4f}")

In [ ]:
# 4. Serialize Model
model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)
with open(os.path.join(model_dir, 'ingredient_model.pkl'), 'wb') as f:
    pickle.dump(ingredient_model, f)
print("Ingredient model saved successfully to models/ingredient_model.pkl")

## 2. AI Model 2: Dish Demand Forecasting & Recommendations (Indian Dishes)

This model predicts the expected number of orders for tomorrow for each dish. It uses historical order volumes and temporal features.

### Features:
- `dish_encoded`: Label encoded dish identifier
- `day_of_week`: 0 (Monday) to 6 (Sunday)
- `month`: 1 to 12
- `lag_1`: Order volume from the previous day (yesterday)
- `lag_7`: Order volume from the same day last week
- `rolling_mean_7`: Average order volume over the last 7 days

In [ ]:
# 1. Generate Daily Order History for 8 Popular Indian Dishes over 180 Days
dishes = [
    "Hyderabadi Chicken Biryani", "Paneer Butter Masala", "Butter Chicken", 
    "Masala Dosa", "Dal Makhani", "Chole Bhature", 
    "Tandoori Chicken", "Gulab Jamun"
]

base_demands = {
    "Hyderabadi Chicken Biryani": 50, "Paneer Butter Masala": 35, "Butter Chicken": 40,
    "Masala Dosa": 45, "Dal Makhani": 25, "Chole Bhature": 30,
    "Tandoori Chicken": 30, "Gulab Jamun": 20
}

dates = pd.date_range(end='2026-07-15', periods=180)
order_records = []

for dish in dishes:
    base = base_demands[dish]
    for date in dates:
        # Weekend effect: higher demand on Friday (4), Saturday (5), Sunday (6)
        day_effect = 1.0
        if date.dayofweek in [4, 5, 6]:
            day_effect = 1.35 if date.dayofweek == 5 else 1.2
        
        # Monthly seasonality (slightly higher demand in winter/monsoon, lower in hot summers)
        month_effect = 1.0
        if date.month in [12, 1, 6, 7]:
            month_effect = 1.1
            
        expected = base * day_effect * month_effect
        noise = np.random.normal(1.0, 0.08)
        qty = int(np.maximum(0, expected * noise))
        
        order_records.append({
            'date': date,
            'dish': dish,
            'orders_count': qty
        })

orders_df = pd.DataFrame(order_records)
orders_df.to_csv('synthetic_orders.csv', index=False)
print("Orders dataset generated successfully! Sample Indian records:")
print(orders_df.head())

In [ ]:
# 2. Feature Engineering for Time-Series Forecasting
orders_df = orders_df.sort_values(by=['dish', 'date'])

# Create Lag and Rolling Mean Features
orders_df['lag_1'] = orders_df.groupby('dish')['orders_count'].shift(1)
orders_df['lag_7'] = orders_df.groupby('dish')['orders_count'].shift(7)
orders_df['rolling_mean_7'] = orders_df.groupby('dish')['orders_count'].shift(1).rolling(7).mean()

# Add Temporal Features
orders_df['day_of_week'] = orders_df['date'].dt.dayofweek
orders_df['month'] = orders_df['date'].dt.month

# Drop NaNs resulting from shifts
orders_df = orders_df.dropna()

# Label Encode the Dish Names
encoder = LabelEncoder()
orders_df['dish_encoded'] = encoder.fit_transform(orders_df['dish'])

print("Feature engineering complete. Sample training record:")
print(orders_df.head())

In [ ]:
# 3. Train Demand Forecasting Model
features = ['dish_encoded', 'day_of_week', 'month', 'lag_1', 'lag_7', 'rolling_mean_7']
X_demand = orders_df[features]
y_demand = orders_df['orders_count']

# Train-Test Split
split_date = pd.to_datetime('2026-06-30')
train_mask = orders_df['date'] <= split_date
test_mask = orders_df['date'] > split_date

X_train_d, X_test_d = X_demand[train_mask], X_demand[test_mask]
y_train_d, y_test_d = y_demand[train_mask], y_demand[test_mask]

demand_model = RandomForestRegressor(n_estimators=50, random_state=42)
demand_model.fit(X_train_d, y_train_d)

# Evaluate
d_predictions = demand_model.predict(X_test_d)
mae_d = mean_absolute_error(y_test_d, d_predictions)
r2_d = r2_score(y_test_d, d_predictions)
print(f"Demand Forecast Model on Test Set -> MAE: {mae_d:.2f} Orders, R^2: {r2_d:.4f}")

In [ ]:
# 4. Serialize Model & Encoder
with open(os.path.join(model_dir, 'demand_model.pkl'), 'wb') as f:
    pickle.dump(demand_model, f)
with open(os.path.join(model_dir, 'dish_encoder.pkl'), 'wb') as f:
    pickle.dump(encoder, f)
print("Demand forecasting model and encoder saved successfully in models/")

## 3. Model Loading & Verification

Let's test loading the saved pickle files and running a test inference.

In [ ]:
# Load files
with open('models/ingredient_model.pkl', 'rb') as f:
    loaded_ingredient_model = pickle.load(f)
with open('models/demand_model.pkl', 'rb') as f:
    loaded_demand_model = pickle.load(f)
with open('models/dish_encoder.pkl', 'rb') as f:
    loaded_encoder = pickle.load(f)

# Test Ingredient Estimation Inference
test_input_sub = pd.DataFrame([[20, 30, 1.2, 0, 0, 1]], columns=['num_subscribers', 'duration_days', 'portion_size_multiplier', 'meal_type_Family Meal', 'meal_type_Protein Meal', 'meal_type_Veg Meal'])
predicted_ingredients = loaded_ingredient_model.predict(test_input_sub)
print("Test Sub Prediction (Veg Meal 20 subs, 30 days, 1.2x portion size):")
for idx, col in enumerate(['Rice', 'Tomato', 'Onion', 'Chicken', 'Paneer']):
    print(f"  {col}: {predicted_ingredients[0][idx]:.2f} kg")

# Test Demand Forecasting Inference (using Indian Dish)
dish_encoded = loaded_encoder.transform(["Hyderabadi Chicken Biryani"])[0]
test_input_demand = pd.DataFrame([[dish_encoded, 4, 7, 52.0, 56.0, 53.0]], columns=['dish_encoded', 'day_of_week', 'month', 'lag_1', 'lag_7', 'rolling_mean_7'])
predicted_demand = loaded_demand_model.predict(test_input_demand)
print(f"Test Demand Prediction (Hyderabadi Chicken Biryani tomorrow): {predicted_demand[0]:.2f} orders")